# YouTube Shorts Pipeline — GUI Fixed

Versi ini menjadikan GUI sebagai entry point utama dan memakai output contract
yang konsisten untuk video, script, metadata, Kora Drive, dan YouTube.

Urutan kerja:

1. Isi konfigurasi.
2. Jalankan semua cell.
3. Buka link AutoShorts GUI yang muncul.
4. Generate video dari GUI.
5. Notebook otomatis melanjutkan setelah MP4 dan metadata valid.

## 1. Konfigurasi

In [ ]:
# =========================================================
# KONFIGURASI UTAMA
# =========================================================

REPO_NAME = "youtube-clipper-photos"
REPO_URL = "https://github.com/hellofadhil/youtube-clipper-photos"
REPO_BRANCH = "main"

GEMINI_API_KEY = ""
PEXELS_API_KEY = ""
PIXABAY_API_KEY = ""
KORA_API_KEY = ""

GEMINI_MODEL = "gemini-3.5-flash-lite"
KORA_API_BASE = "https://api-drive.baselab.web.id"
WATERMARK_TEXT = "@DeepOceanHQ"

RUN_GENERATOR = True
UPLOAD_TO_KORA = True
UPLOAD_TO_YOUTUBE = True
CLEAR_PREVIOUS_OUTPUTS = True

# GUI akan berjalan di background. Cell berikutnya akan menunggu hasil render.
GUI_RENDER_TIMEOUT_MINUTES = 45
MAX_CONCURRENT_WORKERS = 3

# YouTube
YOUTUBE_PRIVACY_STATUS = "private"
YOUTUBE_SCHEDULE_ENABLED = False
YOUTUBE_SCHEDULE_DATE_WIB = "2026-07-29"
YOUTUBE_SCHEDULE_TIME_WIB = "20:30"
YOUTUBE_TOKEN_LINK = ""

## 2. Setup Repository

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

CONTENT_DIR = Path("/content")
REPO_DIR = CONTENT_DIR / REPO_NAME
ASSETS_DIR = REPO_DIR / "assets"
FINAL_SOURCE_DIR = ASSETS_DIR / "final"
PROJECTS_ROOT = CONTENT_DIR / "youtube_shorts_projects"

if not REPO_DIR.exists():
    print("📁 Repository belum ada. Melakukan git clone...")
    subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    print("🔄 Repository sudah ada. Mengambil versi terbaru...")
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{REPO_BRANCH}"],
        check=True,
    )

subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)

requirements_file = REPO_DIR / "requirements.txt"
if requirements_file.exists():
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_file)],
        check=True,
    )

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "gradio",
        "requests",
        "google-api-python-client",
        "google-auth",
        "google-auth-httplib2",
    ],
    check=True,
)

for folder in [
    ASSETS_DIR / "avatar",
    ASSETS_DIR / "audio_clips",
    ASSETS_DIR / "video_clips",
    ASSETS_DIR / "temp",
    ASSETS_DIR / "bgm",
    FINAL_SOURCE_DIR,
    PROJECTS_ROOT,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("✅ Setup repository selesai:", REPO_DIR)

## 3. Environment

In [ ]:
import os
from pathlib import Path

required_values = {
    "GEMINI_API_KEY": GEMINI_API_KEY,
    "PEXELS_API_KEY": PEXELS_API_KEY,
    "WATERMARK_TEXT": WATERMARK_TEXT,
}

invalid_values = [
    name
    for name, value in required_values.items()
    if not value or str(value).startswith("PASTE_")
]

if invalid_values:
    raise ValueError(
        "Konfigurasi belum lengkap: " + ", ".join(invalid_values)
    )

ENV_FILE = REPO_DIR / ".env"
env_lines = [
    f"GEMINI_API_KEY={GEMINI_API_KEY}",
    f"PEXELS_API_KEY={PEXELS_API_KEY}",
    f"PIXABAY_API_KEY={PIXABAY_API_KEY}",
    f"GEMINI_MODEL={GEMINI_MODEL}",
    f"WATERMARK_TEXT={WATERMARK_TEXT}",
    f"MAX_CONCURRENT_WORKERS={MAX_CONCURRENT_WORKERS}",
]

ENV_FILE.write_text("\n".join(env_lines) + "\n", encoding="utf-8")

for name, value in {
    "GEMINI_API_KEY": GEMINI_API_KEY,
    "PEXELS_API_KEY": PEXELS_API_KEY,
    "PIXABAY_API_KEY": PIXABAY_API_KEY,
    "GEMINI_MODEL": GEMINI_MODEL,
    "WATERMARK_TEXT": WATERMARK_TEXT,
    "MAX_CONCURRENT_WORKERS": str(MAX_CONCURRENT_WORKERS),
    "SHORTS_FINAL_DIR": str(FINAL_SOURCE_DIR),
    "PYTHONUNBUFFERED": "1",
}.items():
    os.environ[name] = str(value)

print("✅ Environment siap:", ENV_FILE)

## 4. Pilih Encoder yang Benar-Benar Tersedia

In [ ]:
import os
import shutil
import subprocess


def encoder_works(encoder: str) -> bool:
    ffmpeg = shutil.which("ffmpeg")
    if not ffmpeg:
        return False

    command = [
        ffmpeg,
        "-hide_banner",
        "-loglevel",
        "error",
        "-f",
        "lavfi",
        "-i",
        "color=c=black:s=640x360:r=30:d=0.25",
        "-frames:v",
        "1",
        "-c:v",
        encoder,
        "-pix_fmt",
        "yuv420p",
    ]

    if "nvenc" in encoder:
        command.extend(["-preset", "p4"])

    command.extend(["-f", "null", "-"])

    result = subprocess.run(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        check=False,
    )

    if result.returncode != 0:
        print(f"⚠️ Encoder {encoder} tidak bisa dipakai:")
        print((result.stderr or result.stdout).strip()[-1500:])
        return False

    return True


SELECTED_ENCODER = (
    "h264_nvenc"
    if encoder_works("h264_nvenc")
    else "libx264"
)

os.environ["FFMPEG_VCODEC"] = SELECTED_ENCODER
os.environ["MAX_CONCURRENT_WORKERS"] = str(MAX_CONCURRENT_WORKERS)

print("🎞️ Encoder aktif :", SELECTED_ENCODER)
print("⚡ FFmpeg workers :", MAX_CONCURRENT_WORKERS)

## 5. Pasang Metadata Bridge

In [ ]:
from pathlib import Path

BRIDGE_PATH = REPO_DIR / "gui_metadata_bridge.py"
BRIDGE_PATH.write_text('\n"""Compatibility bridge that persists GUI-generated script metadata.\n\nThis module wraps ContentBrain.generate_script() and writes an atomic output\ncontract to assets/final. It works for both Gradio GUI and CLI entry points.\n"""\n\nfrom __future__ import annotations\n\nimport functools\nimport json\nimport os\nfrom pathlib import Path\nfrom typing import Any\n\n_INSTALLED = False\n_MARKER = "__gui_metadata_bridge_wrapped__"\n\n\ndef _atomic_write_text(path: Path, content: str) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp_path = path.with_name(f".{path.name}.tmp")\n    temp_path.write_text(content, encoding="utf-8")\n    temp_path.replace(path)\n\n\ndef _atomic_write_json(path: Path, payload: Any) -> None:\n    _atomic_write_text(\n        path,\n        json.dumps(payload, indent=2, ensure_ascii=False),\n    )\n\n\ndef _category_name(brain_module: Any, category_key: str) -> str:\n    categories = getattr(brain_module, "TOPIC_CATEGORIES", {})\n    category = categories.get(str(category_key)) if isinstance(categories, dict) else None\n\n    if category is None:\n        return "General Shorts"\n\n    if isinstance(category, dict):\n        return str(\n            category.get("name")\n            or category.get("title")\n            or category_key\n        ).strip()\n\n    return str(\n        getattr(category, "name", None)\n        or getattr(category, "title", None)\n        or category_key\n    ).strip()\n\n\ndef _normalize_hashtags(value: Any) -> str:\n    if isinstance(value, (list, tuple, set)):\n        tags = [str(item).strip() for item in value if str(item).strip()]\n        value = " ".join(tags)\n\n    text = str(value or "").strip()\n    tags = re.findall(r"#[A-Za-z0-9_]+", text)\n\n    if not any(tag.lower() == "#shorts" for tag in tags):\n        tags.insert(0, "#Shorts")\n\n    return " ".join(dict.fromkeys(tags)) or "#Shorts"\n\n\ndef _persist_generation(\n    brain_module: Any,\n    topic: str,\n    category_key: str,\n    script: Any,\n) -> None:\n    if not isinstance(script, dict):\n        return\n\n    final_dir = Path(\n        os.getenv(\n            "SHORTS_FINAL_DIR",\n            str(Path.cwd() / "assets" / "final"),\n        )\n    )\n    final_dir.mkdir(parents=True, exist_ok=True)\n\n    metadata = script.get("metadata")\n    if not isinstance(metadata, dict):\n        metadata = {}\n\n    clean_topic = str(topic or metadata.get("topic") or "Untitled YouTube Short").strip()\n    title = str(metadata.get("title") or clean_topic).strip()\n    description = str(metadata.get("description") or "").strip()\n    hashtags = _normalize_hashtags(metadata.get("hashtags"))\n    category = _category_name(brain_module, category_key)\n\n    metadata_text = (\n        f"TOPIC: {clean_topic}\\n"\n        f"CATEGORY: {category}\\n"\n        f"TITLE: {title}\\n"\n        "DESCRIPTION:\\n"\n        f"{description}\\n"\n        "HASHTAGS:\\n"\n        f"{hashtags}\\n"\n    )\n\n    _atomic_write_json(final_dir / "generated_script.json", script)\n    _atomic_write_text(final_dir / "final_short_metadata.txt", metadata_text)\n    _atomic_write_json(\n        final_dir / "latest_generation.json",\n        {\n            "status": "script_ready",\n            "topic": clean_topic,\n            "category": category,\n            "categoryKey": str(category_key),\n            "title": title,\n            "description": description,\n            "hashtags": hashtags,\n            "metadataPath": str(final_dir / "final_short_metadata.txt"),\n            "scriptPath": str(final_dir / "generated_script.json"),\n        },\n    )\n\n    print(\n        "✅ GUI metadata contract saved:",\n        final_dir / "final_short_metadata.txt",\n    )\n\n\ndef install() -> bool:\n    global _INSTALLED\n\n    if _INSTALLED:\n        return True\n\n    try:\n        from modules import brain as brain_module\n    except Exception as error:\n        print(f"⚠️ GUI metadata bridge could not import modules.brain: {error}")\n        return False\n\n    content_brain = getattr(brain_module, "ContentBrain", None)\n    if content_brain is None:\n        print("⚠️ GUI metadata bridge: ContentBrain was not found.")\n        return False\n\n    original = getattr(content_brain, "generate_script", None)\n    if original is None:\n        print("⚠️ GUI metadata bridge: generate_script() was not found.")\n        return False\n\n    if getattr(original, _MARKER, False):\n        _INSTALLED = True\n        return True\n\n    @functools.wraps(original)\n    def wrapped(self, *args, **kwargs):\n        topic = (\n            args[0]\n            if args\n            else kwargs.get("topic", "Untitled YouTube Short")\n        )\n        category_key = (\n            args[1]\n            if len(args) > 1\n            else kwargs.get("category_key", "1")\n        )\n\n        script = original(self, *args, **kwargs)\n        _persist_generation(\n            brain_module=brain_module,\n            topic=str(topic),\n            category_key=str(category_key),\n            script=script,\n        )\n        return script\n\n    setattr(wrapped, _MARKER, True)\n    content_brain.generate_script = wrapped\n    _INSTALLED = True\n\n    print("✅ GUI metadata bridge installed.")\n    return True\n\n\n# Needed because _normalize_hashtags uses regex.\nimport re\n', encoding="utf-8")

print("✅ GUI metadata bridge ditulis:", BRIDGE_PATH)

## 6. Jalankan AutoShorts GUI

In [ ]:
import os
import re
import shutil
import subprocess
import sys
import time
from pathlib import Path
from IPython.display import HTML, display

GUI_LOG_PATH = Path("/content/autoshorts_gui.log")
GUI_PID_PATH = Path("/content/autoshorts_gui.pid")
LOG_POS = 0

def stream_logs():
    global LOG_POS
    if GUI_LOG_PATH.exists():
        try:
            with GUI_LOG_PATH.open("r", encoding="utf-8", errors="replace") as f:
                f.seek(LOG_POS)
                new_text = f.read()
                if new_text:
                    print(new_text, end="", flush=True)
                    LOG_POS = f.tell()
        except Exception:
            pass

if CLEAR_PREVIOUS_OUTPUTS and FINAL_SOURCE_DIR.exists():
    shutil.rmtree(FINAL_SOURCE_DIR)

FINAL_SOURCE_DIR.mkdir(parents=True, exist_ok=True)

if GUI_PID_PATH.exists():
    try:
        old_pid = int(GUI_PID_PATH.read_text().strip())
        os.kill(old_pid, 0)
        print(f"⚠️ GUI lama masih aktif dengan PID {old_pid}. Menghentikannya...")
        os.kill(old_pid, 15)
        time.sleep(2)
    except Exception:
        pass
    GUI_PID_PATH.unlink(missing_ok=True)

launch_code = (
    "import runpy; "
    "import gui_metadata_bridge; "
    "gui_metadata_bridge.install(); "
    "runpy.run_path('web_app.py', run_name='__main__')"
)

process_env = os.environ.copy()
process_env["SHORTS_FINAL_DIR"] = str(FINAL_SOURCE_DIR)
process_env["PYTHONUNBUFFERED"] = "1"

log_handle = GUI_LOG_PATH.open("w", encoding="utf-8")

GUI_PROCESS = subprocess.Popen(
    [sys.executable, "-u", "-c", launch_code],
    cwd=str(REPO_DIR),
    env=process_env,
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

GUI_PID_PATH.write_text(str(GUI_PROCESS.pid), encoding="utf-8")

public_url = None
deadline = time.time() + 90

while time.time() < deadline:
    stream_logs()
    if GUI_PROCESS.poll() is not None:
        break

    log_text = (
        GUI_LOG_PATH.read_text(encoding="utf-8", errors="replace")
        if GUI_LOG_PATH.exists()
        else ""
    )

    match = re.search(r"https://[A-Za-z0-9.-]+\.gradio\.live", log_text)
    if match:
        public_url = match.group(0)
        break

    time.sleep(1)

stream_logs()

if public_url:
    print("✅ AutoShorts GUI aktif:")
    print(public_url)
    display(
        HTML(
            f'<a href="{public_url}" target="_blank" '
            'style="font-size:18px;font-weight:700;">'
            "🚀 Buka AutoShorts AI Web Studio"
            "</a>"
        )
    )
else:
    print("⚠️ URL publik belum terbaca. Log terbaru:")
    if GUI_LOG_PATH.exists():
        print(GUI_LOG_PATH.read_text(encoding="utf-8", errors="replace")[-4000:])

print()
print("Setelah membuka GUI, pilih kategori/topik lalu klik Generate.")
print("Cell berikutnya akan menunggu sampai video benar-benar selesai.")


## 7. Tunggu dan Finalisasi Hasil GUI

In [ ]:
import hashlib
import json
import re
import shutil
import subprocess
import time
import unicodedata
from datetime import datetime, timezone
from pathlib import Path


def slugify(value: str) -> str:
    normalized = unicodedata.normalize("NFKD", value)
    ascii_value = normalized.encode("ascii", "ignore").decode("ascii")
    result = re.sub(r"[^A-Za-z0-9]+", "_", ascii_value).strip("_")
    return result[:80] or "youtube_short"


def parse_metadata(path: Path) -> dict:
    raw = path.read_text(encoding="utf-8", errors="replace").strip()

    def one_line(label: str, default: str = "") -> str:
        match = re.search(
            rf"(?im)^{re.escape(label)}:\s*(.+?)\s*$",
            raw,
        )
        return match.group(1).strip() if match else default

    description_match = re.search(
        r"(?ims)^DESCRIPTION:\s*(.*?)(?=^\s*HASHTAGS:\s*|\Z)",
        raw,
    )
    description = (
        description_match.group(1).strip()
        if description_match
        else ""
    )

    hashtags_match = re.search(
        r"(?ims)^HASHTAGS:\s*(.*?)\s*$",
        raw,
    )
    hashtags_raw = (
        hashtags_match.group(1).strip()
        if hashtags_match
        else "#Shorts"
    )
    hashtags = list(
        dict.fromkeys(re.findall(r"#[A-Za-z0-9_]+", hashtags_raw))
    )

    if not any(tag.lower() == "#shorts" for tag in hashtags):
        hashtags.insert(0, "#Shorts")

    topic = one_line("TOPIC", "Untitled YouTube Short")
    return {
        "topic": topic,
        "category": one_line("CATEGORY", "General Shorts"),
        "title": one_line("TITLE", topic),
        "description": description,
        "hashtags": hashtags,
        "hashtags_text": " ".join(hashtags),
        "raw": raw,
    }


def video_is_valid(path: Path) -> bool:
    if not path.is_file() or path.stat().st_size < 1_000_000:
        return False

    result = subprocess.run(
        [
            "ffprobe",
            "-v",
            "error",
            "-select_streams",
            "v:0",
            "-show_entries",
            "stream=codec_name,width,height,duration",
            "-of",
            "json",
            str(path),
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        check=False,
    )

    if result.returncode != 0:
        return False

    try:
        payload = json.loads(result.stdout)
        streams = payload.get("streams") or []
        return bool(streams and streams[0].get("codec_name"))
    except Exception:
        return False


if not RUN_GENERATOR:
    print("⏭️ RUN_GENERATOR=False. Menggunakan hasil yang sudah ada.")
else:
    print("⏳ Menunggu hasil render dari GUI...")

deadline = time.time() + (GUI_RENDER_TIMEOUT_MINUTES * 60)
video_source = FINAL_SOURCE_DIR / "final_short.mp4"
metadata_source = FINAL_SOURCE_DIR / "final_short_metadata.txt"

last_size = -1
stable_checks = 0

while time.time() < deadline:
    stream_logs()
    if video_source.exists():
        current_size = video_source.stat().st_size

        if current_size == last_size and current_size > 1_000_000:
            stable_checks += 1
        else:
            stable_checks = 0

        last_size = current_size

        if (
            stable_checks >= 2
            and metadata_source.exists()
            and video_is_valid(video_source)
        ):
            stream_logs()
            break

    time.sleep(3)

stream_logs()
else:
    log_tail = ""
    if GUI_LOG_PATH.exists():
        log_tail = GUI_LOG_PATH.read_text(
            encoding="utf-8",
            errors="replace",
        )[-6000:]

    raise TimeoutError(
        "Render GUI belum menghasilkan bundle valid.\n"
        f"Video    : {video_source} ({video_source.exists()})\n"
        f"Metadata : {metadata_source} ({metadata_source.exists()})\n\n"
        "Log terbaru:\n"
        + log_tail
    )

SHORT_METADATA = parse_metadata(metadata_source)

fingerprint_source = (
    f"{video_source.stat().st_size}:"
    f"{video_source.stat().st_mtime_ns}:"
    f"{SHORT_METADATA['topic']}"
)
project_uid = hashlib.sha256(
    fingerprint_source.encode("utf-8")
).hexdigest()[:10]

PROJECT_DIR = (
    PROJECTS_ROOT
    / f"{slugify(SHORT_METADATA['topic'])}_{project_uid}"
)
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

copy_candidates = [
    "final_short.mp4",
    "final_short_metadata.txt",
    "final_short_preview.mp4",
    "generated_script.json",
    "latest_generation.json",
]

copied_files = []

for name in copy_candidates:
    source = FINAL_SOURCE_DIR / name
    destination = PROJECT_DIR / name

    if not source.exists():
        continue

    shutil.copy2(source, destination)
    copied_files.append(str(destination))

VIDEO_PATH = PROJECT_DIR / "final_short.mp4"
METADATA_PATH = PROJECT_DIR / "final_short_metadata.txt"

manifest = {
    "uid": project_uid,
    "createdAt": datetime.now(timezone.utc).isoformat(),
    "sourceFolder": str(FINAL_SOURCE_DIR),
    "projectFolder": str(PROJECT_DIR),
    "videoPath": str(VIDEO_PATH),
    "metadataPath": str(METADATA_PATH),
    "metadata": {
        "topic": SHORT_METADATA["topic"],
        "category": SHORT_METADATA["category"],
        "title": SHORT_METADATA["title"],
        "description": SHORT_METADATA["description"],
        "hashtags": SHORT_METADATA["hashtags"],
    },
    "files": copied_files,
}

MANIFEST_PATH = PROJECT_DIR / "project_manifest.json"
MANIFEST_PATH.write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

CATEGORY_NAME = slugify(SHORT_METADATA["category"])

print("=" * 70)
print("✅ BUNDLE GUI VALID")
print("Project     :", PROJECT_DIR)
print("Topic       :", SHORT_METADATA["topic"])
print("Category    :", SHORT_METADATA["category"])
print("Title       :", SHORT_METADATA["title"])
print("Hashtags    :", SHORT_METADATA["hashtags_text"])
print("Video       :", VIDEO_PATH)
print("Metadata    :", METADATA_PATH)
print("=" * 70)


## 8. Validasi Video

In [ ]:
import json
import subprocess

video_size_mb = VIDEO_PATH.stat().st_size / 1024 / 1024

result = subprocess.run(
    [
        "ffprobe",
        "-v",
        "error",
        "-show_entries",
        "stream=codec_name,width,height,r_frame_rate,duration",
        "-of",
        "json",
        str(VIDEO_PATH),
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
    check=True,
)

print(f"📦 Ukuran video: {video_size_mb:.2f} MB")
print(json.dumps(json.loads(result.stdout), indent=2))

## 9. Upload ke Kora Drive

In [ ]:
import requests
from pathlib import Path


def upload_to_kora(file_path, target_dir="Storage"):
    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(f"File tidak ditemukan: {file_path}")

    if not KORA_API_KEY or str(KORA_API_KEY).startswith("PASTE_"):
        raise ValueError("KORA_API_KEY belum diisi.")

    file_name = file_path.name
    file_size = file_path.stat().st_size
    headers = {
        "Authorization": f"Bearer {KORA_API_KEY}",
        "Content-Type": "application/json",
    }

    print(f"⬆️ Upload Kora: {file_name} ({file_size / 1024 / 1024:.2f} MB)")

    init_response = requests.post(
        f"{KORA_API_BASE}/files/upload/initiate",
        headers=headers,
        json={"name": file_name, "path": target_dir},
        timeout=60,
    )
    init_response.raise_for_status()
    init_data = init_response.json()

    upload_url = init_data.get("uploadUrl")
    drive_id = (
        init_data.get("driveId")
        or init_data.get("id")
        or init_data.get("fileId")
    )

    if not upload_url:
        raise RuntimeError(f"uploadUrl tidak ditemukan: {init_data}")

    with file_path.open("rb") as stream:
        put_response = requests.put(
            upload_url,
            data=stream,
            headers={"Content-Length": str(file_size)},
            timeout=1800,
        )
    put_response.raise_for_status()

    if put_response.content:
        try:
            put_data = put_response.json()
            drive_id = (
                put_data.get("driveId")
                or put_data.get("id")
                or put_data.get("fileId")
                or drive_id
            )
        except requests.JSONDecodeError:
            pass

    if not drive_id:
        raise RuntimeError("Binary terunggah, tetapi driveId tidak ditemukan.")

    complete_response = requests.post(
        f"{KORA_API_BASE}/files/upload/complete",
        headers=headers,
        json={
            "driveId": drive_id,
            "name": file_name,
            "path": target_dir,
            "type": "file",
            "sizeBytes": file_size,
        },
        timeout=60,
    )
    complete_response.raise_for_status()

    print(f"✅ Berhasil upload ke Kora: {file_name}")
    try:
        return complete_response.json()
    except requests.JSONDecodeError:
        return {"success": True, "driveId": drive_id}


KORA_TARGET_FOLDER = (
    f"Video/Shorts/{CATEGORY_NAME}/{PROJECT_DIR.name}"
)
KORA_UPLOAD_RESULTS = []

if UPLOAD_TO_KORA:
    for file_path in [
        VIDEO_PATH,
        METADATA_PATH,
        PROJECT_DIR / "generated_script.json",
        MANIFEST_PATH,
    ]:
        if not file_path.exists():
            print(f"⚠️ Dilewati: {file_path.name}")
            continue

        try:
            result = upload_to_kora(file_path, KORA_TARGET_FOLDER)
            KORA_UPLOAD_RESULTS.append(
                {"file": file_path.name, "success": True, "result": result}
            )
        except Exception as error:
            print(f"❌ Gagal upload {file_path.name}: {error}")
            KORA_UPLOAD_RESULTS.append(
                {"file": file_path.name, "success": False, "error": str(error)}
            )
else:
    print("⏭️ Upload Kora dilewati.")

## 10. Autentikasi YouTube

In [ ]:
import json
import time
import requests
from pathlib import Path

from google.colab import files
from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request as GoogleAuthRequest
from google.auth.exceptions import RefreshError
from googleapiclient.discovery import build

TOKEN_FILE = Path("/content/youtube_token.json")

SCOPES = [
    "https://www.googleapis.com/auth/youtube.upload"
]

DEVICE_CODE_URL = "https://oauth2.googleapis.com/device/code"
DEFAULT_TOKEN_URI = "https://oauth2.googleapis.com/token"


def validate_token_file(token_path: Path) -> dict:
    with token_path.open("r", encoding="utf-8") as file:
        token_data = json.load(file)

    if not isinstance(token_data, dict):
        raise ValueError("Isi token bukan JSON object.")

    required_fields = [
        "client_id",
        "client_secret",
        "token_uri",
    ]

    missing_fields = [
        field
        for field in required_fields
        if not token_data.get(field)
    ]

    if missing_fields:
        raise ValueError(
            "Token tidak lengkap. Field hilang: "
            + ", ".join(missing_fields)
        )

    return token_data


def download_youtube_token() -> bool:
    if TOKEN_FILE.exists():
        try:
            validate_token_file(TOKEN_FILE)
            print("✅ Token lokal sudah tersedia.")
            return True
        except Exception:
            print("⚠️ Token lokal rusak. Mengunduh ulang...")
            TOKEN_FILE.unlink(missing_ok=True)

    if (
        not YOUTUBE_TOKEN_LINK
        or YOUTUBE_TOKEN_LINK.startswith("PASTE_")
        or not YOUTUBE_TOKEN_LINK.startswith(("http://", "https://"))
    ):
        print("⚠️ YOUTUBE_TOKEN_LINK belum valid.")
        return False

    print("⬇️ Mengunduh youtube_token.json...")

    try:
        response = requests.get(
            YOUTUBE_TOKEN_LINK,
            timeout=60,
            allow_redirects=True,
        )
        response.raise_for_status()

        content_type = response.headers.get(
            "Content-Type",
            "",
        ).lower()

        if "text/html" in content_type:
            raise RuntimeError(
                "Link menghasilkan HTML, bukan JSON. "
                "Gunakan direct-download link."
            )

        TOKEN_FILE.write_bytes(response.content)
        validate_token_file(TOKEN_FILE)

        print("✅ Token berhasil didownload:", TOKEN_FILE)
        return True

    except Exception as error:
        TOKEN_FILE.unlink(missing_ok=True)
        print("⚠️ Gagal download token:", error)
        return False


def load_saved_credentials():
    if not TOKEN_FILE.exists():
        return None

    try:
        credentials = Credentials.from_authorized_user_file(
            str(TOKEN_FILE),
            SCOPES,
        )

        if credentials.expired and credentials.refresh_token:
            print("🔄 Memperbarui access token...")
            credentials.refresh(GoogleAuthRequest())

            TOKEN_FILE.write_text(
                credentials.to_json(),
                encoding="utf-8",
            )

            print("✅ Access token berhasil diperbarui.")

        if credentials.valid:
            print("✅ Menggunakan token YouTube tersimpan.")
            return credentials

        print("⚠️ Token YouTube tidak valid.")

    except RefreshError as error:
        print("⚠️ Refresh token ditolak atau dicabut:", error)

    except Exception as error:
        print("⚠️ Gagal membaca token:", error)

    TOKEN_FILE.unlink(missing_ok=True)
    return None


def upload_oauth_json() -> dict:
    print()
    print("=" * 65)
    print("TOKEN TIDAK TERSEDIA ATAU TIDAK VALID")
    print("Upload OAuth Client JSON bertipe TVs and Limited Input Devices.")
    print("=" * 65)

    uploaded_oauth = files.upload()

    oauth_files = [
        Path(filename)
        for filename in uploaded_oauth.keys()
        if filename.lower().endswith(".json")
        and filename != TOKEN_FILE.name
    ]

    if not oauth_files:
        raise FileNotFoundError(
            "OAuth Client JSON tidak ditemukan."
        )

    oauth_path = oauth_files[0]

    with oauth_path.open("r", encoding="utf-8") as file:
        oauth_config = json.load(file)

    client_config = oauth_config.get("installed")

    if not client_config:
        raise ValueError(
            "OAuth Client tidak cocok. Gunakan tipe "
            "'TVs and Limited Input devices'."
        )

    return client_config


def login_with_device_flow(client_config: dict):
    client_id = client_config["client_id"]
    client_secret = client_config.get("client_secret", "")
    token_uri = client_config.get("token_uri", DEFAULT_TOKEN_URI)

    device_response = requests.post(
        DEVICE_CODE_URL,
        data={
            "client_id": client_id,
            "scope": " ".join(SCOPES),
        },
        timeout=30,
    )

    if not device_response.ok:
        raise RuntimeError(
            "Gagal meminta device code:\n"
            + device_response.text
        )

    device_data = device_response.json()

    verification_url = (
        device_data.get("verification_url")
        or device_data.get("verification_uri")
        or device_data.get("verification_uri_complete")
    )

    print()
    print("=" * 65)
    print("LOGIN YOUTUBE DIPERLUKAN")
    print("1. Buka:", verification_url)
    print("2. Masukkan kode:", device_data["user_code"])
    print("=" * 65)

    interval = int(device_data.get("interval", 5))
    expires_at = time.time() + int(device_data["expires_in"])

    while time.time() < expires_at:
        token_response = requests.post(
            token_uri,
            data={
                "client_id": client_id,
                "client_secret": client_secret,
                "device_code": device_data["device_code"],
                "grant_type": (
                    "urn:ietf:params:oauth:"
                    "grant-type:device_code"
                ),
            },
            timeout=30,
        )

        token_data = token_response.json()

        if token_response.ok:
            credentials = Credentials(
                token=token_data["access_token"],
                refresh_token=token_data.get("refresh_token"),
                token_uri=token_uri,
                client_id=client_id,
                client_secret=client_secret,
                scopes=SCOPES,
            )

            TOKEN_FILE.write_text(
                credentials.to_json(),
                encoding="utf-8",
            )

            print("✅ Login berhasil.")
            print("✅ Token baru disimpan di:", TOKEN_FILE)
            return credentials

        error_code = token_data.get("error")

        if error_code == "authorization_pending":
            time.sleep(interval)
            continue

        if error_code == "slow_down":
            interval += 5
            time.sleep(interval)
            continue

        if error_code == "access_denied":
            raise RuntimeError("Login atau akses ditolak.")

        if error_code == "expired_token":
            raise RuntimeError(
                "Kode login kedaluwarsa. Jalankan ulang cell."
            )

        raise RuntimeError(f"OAuth error: {token_data}")

    raise TimeoutError("Waktu autentikasi habis.")


def authenticate_youtube():
    download_youtube_token()

    credentials = load_saved_credentials()

    if credentials:
        return credentials

    client_config = upload_oauth_json()
    return login_with_device_flow(client_config)


youtube = None
credentials = None

if UPLOAD_TO_YOUTUBE:
    credentials = authenticate_youtube()

    youtube = build(
        "youtube",
        "v3",
        credentials=credentials,
        cache_discovery=False,
    )

    print("✅ YouTube API berhasil terhubung.")
else:
    print("⏭️ UPLOAD_TO_YOUTUBE=False. Autentikasi YouTube dilewati.")

## 11. Upload ke YouTube Shorts

In [ ]:
import os
import time
import random

from datetime import datetime, timezone, timedelta
from zoneinfo import ZoneInfo

from googleapiclient.http import MediaFileUpload
from googleapiclient.errors import HttpError


def upload_youtube_video(
    video_path,
    title,
    description,
    tags=None,
    category_id="27",
    privacy_status="private",
    made_for_kids=False,
    schedule_enabled=False,
    schedule_date_wib=None,
    schedule_time_wib=None,
):
    if youtube is None:
        raise RuntimeError(
            "YouTube API belum terhubung."
        )

    if not os.path.isfile(video_path):
        raise FileNotFoundError(
            f"File video tidak ditemukan: {video_path}"
        )

    allowed_privacy = {
        "private",
        "unlisted",
        "public",
    }

    if privacy_status not in allowed_privacy:
        raise ValueError(
            "Privacy harus private, unlisted, atau public."
        )

    publish_schedule = None

    if schedule_enabled:
        if not schedule_date_wib:
            raise ValueError(
                "Tanggal schedule YouTube belum diisi."
            )

        if not schedule_time_wib:
            raise ValueError(
                "Jam schedule YouTube belum diisi."
            )

        if privacy_status != "private":
            raise ValueError(
                "Scheduled upload wajib menggunakan "
                "privacy_status='private'."
            )

        publish_schedule = create_youtube_publish_time(
            schedule_date_wib=schedule_date_wib,
            schedule_time_wib=schedule_time_wib,
        )

    status_body = {
        "privacyStatus": privacy_status,
        "selfDeclaredMadeForKids": made_for_kids,
    }

    if publish_schedule:
        status_body["publishAt"] = publish_schedule[
            "utc_text"
        ]

    request_body = {
        "snippet": {
            "title": title[:100],
            "description": description[:5000],
            "tags": (tags or [])[:500],
            "categoryId": str(category_id),
        },
        "status": status_body,
    }

    print()
    print("=" * 70)
    print("📤 PERSIAPAN UPLOAD YOUTUBE")
    print("=" * 70)
    print("Video       :", video_path)
    print("Title       :", title[:100])
    print("Privacy     :", privacy_status)

    if publish_schedule:
        print(
            "Publish WIB :",
            publish_schedule["wib_text"],
        )
        print(
            "Publish UTC :",
            publish_schedule["utc_text"],
        )
        print("Mode        : Scheduled")
    else:
        print("Mode        : Upload langsung")

    print("=" * 70)

    media = MediaFileUpload(
        video_path,
        mimetype="video/mp4",
        chunksize=8 * 1024 * 1024,
        resumable=True,
    )

    request = youtube.videos().insert(
        part="snippet,status",
        body=request_body,
        media_body=media,
        notifySubscribers=False,
    )

    response = None
    retry_count = 0
    max_retries = 5

    print("🚀 Mulai upload ke YouTube...")

    while response is None:
        try:
            upload_status, response = request.next_chunk()

            if upload_status:
                progress = int(
                    upload_status.progress() * 100
                )
                print(f"Upload: {progress}%")

        except HttpError as error:
            retryable_errors = {
                500,
                502,
                503,
                504,
            }

            status_code = getattr(
                error.resp,
                "status",
                None,
            )

            if (
                status_code in retryable_errors
                and retry_count < max_retries
            ):
                delay = (
                    2 ** retry_count
                ) + random.random()

                print(
                    f"Server error {status_code}. "
                    f"Retry dalam {delay:.1f} detik..."
                )

                time.sleep(delay)
                retry_count += 1
                continue

            print("❌ Upload gagal:", error)
            raise

    video_id = response["id"]
    video_url = (
        f"https://www.youtube.com/watch?v={video_id}"
    )

    print()
    print("=" * 70)
    print("✅ UPLOAD YOUTUBE BERHASIL")
    print("=" * 70)
    print("Video ID :", video_id)
    print("URL      :", video_url)
    print("Privacy  :", privacy_status)

    if publish_schedule:
        print("Status   : Scheduled")
        print(
            "Tayang    :",
            publish_schedule["wib_text"],
        )
    else:
        print("Status   :", privacy_status)

    print("=" * 70)

    return {
        "response": response,
        "videoId": video_id,
        "videoUrl": video_url,
        "privacyStatus": privacy_status,
        "isScheduled": publish_schedule is not None,
        "scheduledPublishAtWIB": (
            publish_schedule["wib_text"]
            if publish_schedule
            else None
        ),
        "scheduledPublishAtUTC": (
            publish_schedule["utc_text"]
            if publish_schedule
            else None
        ),
    }


# ============================================================
# MENYIAPKAN METADATA DAN UPLOAD
# ============================================================

YOUTUBE_UPLOAD_RESULT = None

if UPLOAD_TO_YOUTUBE:
    youtube_description_parts = []

    description_text = SHORT_METADATA.get(
        "description",
        "",
    ).strip()

    hashtags_text = SHORT_METADATA.get(
        "hashtags_text",
        "",
    ).strip()

    if description_text:
        youtube_description_parts.append(
            description_text
        )

    if hashtags_text:
        youtube_description_parts.append(
            hashtags_text
        )

    youtube_description = "\n\n".join(
        youtube_description_parts
    )

    youtube_tags = [
        hashtag.lstrip("#").strip()
        for hashtag in SHORT_METADATA.get(
            "hashtags",
            [],
        )
        if hashtag.strip()
    ]

    print()
    print("=" * 70)
    print("📝 METADATA YOUTUBE")
    print("=" * 70)
    print(
        "Title       :",
        SHORT_METADATA["title"],
    )
    print(
        "Description :",
        youtube_description,
    )
    print(
        "Tags        :",
        youtube_tags,
    )

    if YOUTUBE_SCHEDULE_ENABLED:
        print(
            "Schedule    :",
            YOUTUBE_SCHEDULE_DATE_WIB,
            YOUTUBE_SCHEDULE_TIME_WIB,
            "WIB",
        )
    else:
        print(
            "Schedule    : Tidak menggunakan jadwal"
        )

    print("=" * 70)

    YOUTUBE_UPLOAD_RESULT = upload_youtube_video(
        video_path=str(VIDEO_PATH),
        title=SHORT_METADATA["title"],
        description=youtube_description,
        tags=youtube_tags,
        category_id="27",
        privacy_status=YOUTUBE_PRIVACY_STATUS,
        made_for_kids=False,
        schedule_enabled=YOUTUBE_SCHEDULE_ENABLED,
        schedule_date_wib=(
            YOUTUBE_SCHEDULE_DATE_WIB
        ),
        schedule_time_wib=(
            YOUTUBE_SCHEDULE_TIME_WIB
        ),
    )

else:
    print("⏭️ Upload YouTube dilewati.")

## 12. Ringkasan

In [ ]:
print("=" * 70)
print("📦 RINGKASAN PIPELINE GUI")
print("=" * 70)
print("Project folder :", PROJECT_DIR)
print("Video          :", VIDEO_PATH)
print("Topic          :", SHORT_METADATA["topic"])
print("Category       :", SHORT_METADATA["category"])
print("Title          :", SHORT_METADATA["title"])
print("Hashtags       :", SHORT_METADATA["hashtags_text"])
print("Encoder        :", SELECTED_ENCODER)

if UPLOAD_TO_KORA:
    print(
        "Kora success   :",
        sum(1 for item in KORA_UPLOAD_RESULTS if item["success"]),
    )
    print(
        "Kora failed    :",
        sum(1 for item in KORA_UPLOAD_RESULTS if not item["success"]),
    )
    print("Kora folder    :", KORA_TARGET_FOLDER)
else:
    print("Kora           : dilewati")

if YOUTUBE_UPLOAD_RESULT:
    print("YouTube URL    :", YOUTUBE_UPLOAD_RESULT["videoUrl"])
    print(
        "YouTube status :",
        "scheduled"
        if YOUTUBE_UPLOAD_RESULT["isScheduled"]
        else YOUTUBE_UPLOAD_RESULT["privacyStatus"],
    )
else:
    print("YouTube        : dilewati / belum berhasil")

print("=" * 70)